# 03 · Ajuste fino contrastivo (InfoNCE)

Ajusta os últimos blocos do DINOv2 com pares (mapa pré-evento, variante de observação pós-evento) da mesma célula, **só na faixa de treino da dobra** do Taquari. Checkpoint no Drive a cada ≤ 15 min; se o tempo do job acabar, o `DONE.json` sai com `FAILED` e é só rodar de novo (retoma). No fim grava `MyDrive/drone_v3/finetune/<job_id>/backbone.pt`.

Antes: rode `00_setup.ipynb`. O job é **retomável**: se a sessão cair, rode o setup e este notebook de novo com o mesmo `JOB_ID`.

In [ ]:
JOB_ID = 'finetune_A'
import pandas as pd
jobs = pd.read_csv('/content/repo/colab/jobs_v3.csv')
job = jobs.set_index('job_id').loc[JOB_ID]
JOB_CONFIG = job.config_path
print(job)

In [ ]:
import subprocess, sys
cmd = [sys.executable, '-m', 'dal.v3.descriptors.finetune', '--job-config', JOB_CONFIG, '--job-id', JOB_ID,
       '--out-root', '/content/drive/MyDrive/drone_v3', '--colab-out', 'colab_out']
print(subprocess.run(cmd, cwd='/content/repo', check=True).returncode)

In [ ]:
import json
done = json.load(open(f'/content/repo/colab_out/{JOB_ID}/DONE.json'))
print(json.dumps(done, indent=1))
main_number = str(done['metrics'].get('crops_per_second', ''))[:12]
subprocess.run([sys.executable, 'scripts/colab_publish_v3.py', '--job-id', JOB_ID,
                '--main-number', f'crops/s {main_number}'], cwd='/content/repo', check=True)